In [ ]:
!wget -P /home/work/data http://easydl-download.bj.bcebos.com/notebook-dataset/fileDownload && chmod +x /home/work/data/fileDownload && /home/work/data/fileDownload -token=G4juMkAyI2KnS46BELjaE68otTmEui7PsHctgVXLigsau+WnctPTsEXDWFjgb8OFCF3k5cEHq78XqnnH+epKWpTSTHXvTBMeOPfyKIzhvyX4DYzXGcNxI6PlvNEyc2sQqIOiSa4Dxp1uJblpdzEFeBD2qkIJzbloGNxmxZpZ9dB8WjjcSePq0ixymX0yq1z+KH3vsUDHqdHiCoRSc4Z7H26TbVcSvv8hirJQXH0x+CAbMn/z6W9eRQz21+koLpmB4oZ1a4kKmdYj06pNyi/5wzE0cyGA60vvshNU4LHQVYxLC7KjNtudpTV914TcLP5VswWpitBtLvXQ+YzVgGkOkKg5T194rsn7CObnqI4rQY8=

--2023-02-18 12:04:56--  http://easydl-download.bj.bcebos.com/notebook-dataset/fileDownload
Resolving easydl-download.bj.bcebos.com (easydl-download.bj.bcebos.com)... 182.61.200.229, 182.61.200.195, 2409:8c04:1001:1002:0:ff:b001:368a
Connecting to easydl-download.bj.bcebos.com (easydl-download.bj.bcebos.com)|182.61.200.229|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6626563 (6.3M) [application/octet-stream]
Saving to: '/home/work/data/fileDownload'

fileDownload        100%[===================>]   6.32M  --.-KB/s    in 0.1s    

2023-02-18 12:04:56 (43.8 MB/s) - '/home/work/data/fileDownload' saved [6626563/6626563]

Downloading... 2.3                                                             omplete                            GB completeGB completeGB complete                    GB complete                    GB completeGB complete                                omplete                        GB complete    GB complete        GB completeGB complete       

In [ ]:
!unzip data/rc-upload-1676692049146-2-0000.Data.zip

Archive:  data/rc-upload-1676692049146-2-0000.Data.zip
   creating: Data/
  inflating: Data/eval.txt           
   creating: Data/evalImageSet/
  inflating: Data/evalImageSet/IM-0001-0001.jpeg  
  inflating: Data/evalImageSet/IM-0003-0001.jpeg  
  inflating: Data/evalImageSet/IM-0005-0001.jpeg  
  inflating: Data/evalImageSet/IM-0006-0001.jpeg  
  inflating: Data/evalImageSet/IM-0007-0001.jpeg  
  inflating: Data/evalImageSet/IM-0009-0001.jpeg  
  inflating: Data/evalImageSet/IM-0010-0001.jpeg  
  inflating: Data/evalImageSet/IM-0011-0001-0001.jpeg  
  inflating: Data/evalImageSet/IM-0011-0001-0002.jpeg  
  inflating: Data/evalImageSet/IM-0011-0001.jpeg  
  inflating: Data/evalImageSet/IM-0013-0001.jpeg  
  inflating: Data/evalImageSet/IM-0015-0001.jpeg  
  inflating: Data/evalImageSet/IM-0016-0001.jpeg  
  inflating: Data/evalImageSet/IM-0017-0001.jpeg  
  inflating: Data/evalImageSet/IM-0019-0001.jpeg  
  inflating: Data/evalImageSet/IM-0021-0001.jpeg  
  inflating: Data/evalImageSet

In [1]:
!pip install ml_collections

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     |████████████████████████████████| 81kB 4.1MB/s eta 0:00:011
  Created wheel for ml-collections: filename=ml_collections-0.1.1-cp37-none-any.whl size=94510 sha256=bcd797c4a5b581963003757d2a33e2378889ab3b934d42357820b6b44ccf4c4c
  Stored in directory: /home/work/.cache/pip/wheels/dd/7a/c2/dd1204c5e04fc8cfb0dc705daa96b65ab405daf8c2eeec76e0
Successfully built ml-collections


In [2]:
!pip install einops

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     |████████████████████████████████| 51kB 235kB/s eta 0:00:011


In [7]:
!pip install timm

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [8]:
import timm

ImportError: cannot import name '_compile_and_register_class' from 'torch.jit._recursive' (/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/torch/jit/_recursive.py)

In [22]:
!pip install tqdm

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [29]:
import codecs
import os
import shutil
from PIL import Image
import csv
import torch
from torch.nn.modules import *
from functools import partial
import math
import time
import timm
"""all_file_dir='./Data'
train_file_dir = './Data/train'
test_file_dir='./Data/test'
val_file_dir='./Data/val'
class_list = [c for c in os.listdir(train_file_dir) if os.path.isdir(os.path.join(train_file_dir, c)) and not c.endswith('Set') and not c.startswith('.')]
class_list.sort()
print(class_list)
train_image_dir = os.path.join(all_file_dir, "trainImageSet")
if not os.path.exists(train_image_dir):
    os.makedirs(train_image_dir)
    
test_image_dir = os.path.join(all_file_dir, "testImageSet")
if not os.path.exists(test_image_dir):
    os.makedirs(test_image_dir)

val_image_dir = os.path.join(all_file_dir, "valImageSet")
if not os.path.exists(val_image_dir):
    os.makedirs(val_image_dir)
    
train_file = open("./Data/train.csv", 'w')
test_file = open(os.path.join(all_file_dir, "test.csv"), 'w')
val_file = open(os.path.join(all_file_dir, "val.csv"), 'w')

with codecs.open(os.path.join(all_file_dir, "label_list.csv"), "w") as label_list:
    #加入训练集
    label_id = 0
    for class_dir in class_list:
        csv.writer(label_list).writerow([label_id, class_dir])
        image_path_pre = os.path.join(train_file_dir, class_dir)
                # 存在一些文件打不开，此处需要稍作清洗
        label_id += 1
    label_id = 0
    for class_dir in class_list:
        print("1")
        image_path_pre = os.path.join(train_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(train_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(train_file).writerow([os.path.join(train_image_dir, file), label_id])
            except Exception as e:
                pass
    ##加入测试集
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(test_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(test_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(test_file).writerow([os.path.join(test_image_dir, file), label_id])
            except Exception as e:
                pass
                # 存在一些文件打不开，此处需要稍作清洗
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(val_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(val_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(val_file).writerow([os.path.join(val_image_dir, file), label_id])
            except Exception as e:
                pass
            
train_file.close()
test_file.close()  
val_file.close()  """
from timm.models.vision_transformer import vit_base_patch16_224_in21k as create_model  
import torch
import pandas as pd
import numpy as np
import os
from PIL import Image
from torch.utils.data import Dataset
from torch import nn
from torch.nn import Conv2d
from torch.nn import Dropout
import copy 

class MyDataSet(Dataset):
    def __init__(self,image_path,csv_path,transforms,phrase):
        # 读取csv
        csv=pd.read_csv(csv_path,header=None)
        # 读取第一列,组合成完整的图片地址
        self.imgs=[str(k) for k in csv[0].values]
        self.phrase=phrase
        if self.phrase!="test":
            self.labels=np.asarray([k for k in csv[1].values])
        self.transforms=transforms
    

    def __getitem__(self, index):
        img_path = self.imgs[index]
        pil_img = Image.open(img_path).convert("RGB")
        if self.transforms:
            data = self.transforms(pil_img)
        else:
            pil_img = np.asarray(pil_img)
            data = torch.from_numpy(pil_img)
        if self.phrase!="test":
            label=self.labels[index]
            sample = (data,label)
        else:
            sample=data
        return sample

    def __len__(self):
        return len(self.imgs)
from torchvision import transforms
# 数据增强
data_transforms={
    "train":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "val":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "test":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    
}

class Embeddings(nn.Module):
    '''
    对图像进行编码，把图片当做一个句子，把图片分割成块，每一块表示一个单词
    '''
    def __init__(self,config,img_size,in_channels=3):
        super(Embeddings,self).__init__()
        img_size=img_size#224
        patch_size=config.patches["size"]#16
        ##将图片分割成多少块（224/16）*（224/16）=196
        n_patches=(img_size//patch_size)*(img_size//patch_size)
        #对图片进行卷积获取图片的块，并且将每一块映射成config.hidden_size维（768）
        self.patch_embeddings=Conv2d(in_channels=in_channels,
                                     out_channels=config.hidden_size,
                                     kernel_size=patch_size,
                                     stride=patch_size)
        
        #设置可学习的位置编码信息，（1,196+1,786）
        self.position_embeddings=nn.Parameter(torch.zeros(1,
                                                          n_patches+1,
                                                          config.hidden_size))
        #设置可学习的分类信息的维度
        self.classifer_token=nn.Parameter(torch.zeros(1,1,config.hidden_size))
        self.dropout=Dropout((config.transformer["dropout_rate"]))

    def forward(self,x):
        bs=x.shape[0]
        #cls_tokens=self.classifer_token.expand(bs,-1,-1)(bs,1,768)
        x=self.patch_embeddings(x)#（bs,768,14,14）
        x=x.flatten(2)#(bs,768,196)
        x=x.transpose(-1,-2)#(bs,196,768)
        cls_tokens=self.classifer_token.expand(bs,-1,-1)
        x=torch.cat((cls_tokens,x),dim=1)#将分类信息与图片块进行拼接（bs,197,768）
        embeddings=x+self.position_embeddings#将图片块信息和对其位置信息进行相加(bs,197,768)
        embeddings=self.dropout(embeddings)
        return  embeddings
# --------------------------------------- #
#（1）patch embedding
'''
img_size=224 : 输入图像的宽高
patch_size=16 ： 每个patch的宽高，也是卷积核的尺寸和步长
in_c=3 ： 输入图像的通道数
embed_dim=768 ： 卷积输出通道数
'''
# --------------------------------------- #
class patchembed(nn.Module):
    # 初始化
    def __init__(self, img_size=224, patch_size=16, in_c=3, embed_dim=768):
        super(patchembed, self).__init__()
        
        # 输入图像的尺寸224*224
        self.img_size = (img_size, img_size)
        # 每个patch的大小16*16
        self.patch_size = (patch_size, patch_size)
        # 将输入图像划分成14*14个patch
        self.grid_size = (img_size//patch_size, img_size//patch_size)
        # 一共有14*14个patch
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        
        # 使用16*16的卷积切分图像，将图像分成14*14个
        self.proj = nn.Conv2d(in_channels=in_c, out_channels=embed_dim, 
                              kernel_size=patch_size, stride=patch_size)
        
        # 定义标准化方法，给LN传入默认参数eps
        norm_layer = partial(nn.LayerNorm, eps=1e-6)
        self.norm = norm_layer(embed_dim)
        
        
    # 前向传播
    def forward(self, inputs):
        # 获得输入图像的shape
        B, C, H, W = inputs.shape
        
        # 如果输入图像的宽高不等于224*224就报错
        assert H==self.img_size[0] and W==self.img_size[1], 'input shape does not match 224*224'
        
        # 卷积层切分patch [b,3,224,224]==>[b,768,14,14]
        x = self.proj(inputs)
        # 展平 [b,768,14,14]==>[b,768,14*14]
        x = x.flatten(start_dim=2, end_dim=-1)  # 将索引为 start_dim 和 end_dim 之间（包括该位置）的数量相乘
        # 维度调整 [b,768,14*14]==>[b,14*14,768]
        x = x.transpose(1, 2)  # 实现一个张量的两个轴之间的维度转换
        # 标准化
        x = self.norm(x)
        
        return x

#2.构建self-Attention模块
class Attention(nn.Module):
    def __init__(self,config,vis):
        super(Attention,self).__init__()
        self.vis=vis
        self.num_attention_heads=config.transformer["num_heads"]#12
        self.attention_head_size = int(config.hidden_size / self.num_attention_heads)  # 768/12=64
        self.all_head_size = self.num_attention_heads * self.attention_head_size  # 12*64=768

        self.query = Linear(config.hidden_size, self.all_head_size)#wm,768->768，Wq矩阵为（768,768）
        self.key = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wk矩阵为（768,768）
        self.value = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wv矩阵为（768,768）
        self.out = Linear(config.hidden_size, config.hidden_size)  # wm,768->768
        self.attn_dropout = Dropout(config.transformer["attention_dropout_rate"])
        self.proj_dropout = Dropout(config.transformer["attention_dropout_rate"])

        self.softmax = Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (
        self.num_attention_heads, self.attention_head_size)  # wm,(bs,197)+(12,64)=(bs,197,12,64)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)  # wm,(bs,12,197,64)

    def forward(self, hidden_states):
        # hidden_states为：(bs,197,768)
        mixed_query_layer = self.query(hidden_states)#wm,768->768
        mixed_key_layer = self.key(hidden_states)#wm,768->768
        mixed_value_layer = self.value(hidden_states)#wm,768->768

        query_layer = self.transpose_for_scores(mixed_query_layer)#wm，(bs,12,197,64)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))#将q向量和k向量进行相乘（bs,12,197,197)
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)#将结果除以向量维数的开方
        attention_probs = self.softmax(attention_scores)#将得到的分数进行softmax,得到概率
        weights = attention_probs if self.vis else None#wm,实际上就是权重
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)#将概率与内容向量相乘
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)#wm,(bs,197)+(768,)=(bs,197,768)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights#wm,(bs,197,768),(bs,197,197)

#3.构建前向传播神经网络
#两个全连接神经网络，中间加了激活函数
class Mlp(nn.Module):
    def __init__(self, config):
        super(Mlp, self).__init__()
        self.fc1 = Linear(config.hidden_size, config.transformer["mlp_dim"])#wm,786->3072
        self.fc2 = Linear(config.transformer["mlp_dim"], config.hidden_size)#wm,3072->786
        self.act_fn = torch.nn.functional.gelu#wm,激活函数
        self.dropout = Dropout(config.transformer["dropout_rate"])

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.normal_(self.fc1.bias, std=1e-6)
        nn.init.normal_(self.fc2.bias, std=1e-6)

    def forward(self, x):
        x = self.fc1(x)#wm,786->3072
        x = self.act_fn(x)#激活函数
        x = self.dropout(x)#wm,丢弃
        x = self.fc2(x)#wm3072->786
        x = self.dropout(x)
        return x

#4.构建编码器的可重复利用的Block()模块：每一个block包含了self-attention模块和MLP模块
class Block(nn.Module):
    def __init__(self, config, vis):
        super(Block, self).__init__()
        self.hidden_size = config.hidden_size#wm,768
        self.attention_norm = LayerNorm(config.hidden_size, eps=1e-6)#wm，层归一化
        self.ffn_norm = LayerNorm(config.hidden_size, eps=1e-6)
        
        self.ffn = Mlp(config)
        self.attn = Attention(config, vis)

    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x = x + h#残差结构

        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x = x + h#残差结构
        return x, weights

#5.构建Encoder模块，该模块实际上就是堆叠N个Block模块
class Encoder(nn.Module):
    def __init__(self, config, vis):
        super(Encoder, self).__init__()
        self.vis = vis
        self.layer = nn.ModuleList()
        self.encoder_norm = LayerNorm(config.hidden_size, eps=1e-6)
        for _ in range(config.transformer["num_layers"]):
            layer = Block(config, vis)
            self.layer.append(copy.deepcopy(layer))

    def forward(self, hidden_states):
        attn_weights = []
        for layer_block in self.layer:
            hidden_states, weights = layer_block(hidden_states)
            if self.vis:
                attn_weights.append(weights)
        encoded = self.encoder_norm(hidden_states)
        return encoded, attn_weights

#6构建transformers完整结构，首先图片被embedding模块编码成序列数据，然后送入Encoder中进行编码
class Transformer(nn.Module):
    def __init__(self, config, img_size, vis):
        super(Transformer, self).__init__()
        self.embeddings = Embeddings(config, img_size=img_size)#wm,对一幅图片进行切块编码，得到的是（bs,n_patch+1（196）,每一块的维度（768））
        self.encoder = Encoder(config, vis)

    def forward(self, input_ids):
        embedding_output = self.embeddings(input_ids)#wm,输出的是（bs,196,768)
        encoded, attn_weights = self.encoder(embedding_output)#wm,输入的是（bs,196,768)
        return encoded, attn_weights#输出的是（bs,197,768）

#7构建VisionTransformer，用于图像分类
class VisionTransformer(nn.Module):
    def __init__(self, config, img_size=224, num_classes=21843, zero_head=False, vis=False):
        super(VisionTransformer, self).__init__()
        self.num_classes = num_classes
        self.zero_head = zero_head
        self.classifier = config.classifier

        self.transformer = Transformer(config, img_size, vis)
        self.head = Linear(config.hidden_size, num_classes)#wm,768-->10

    def forward(self, x, labels=None):
        x, attn_weights = self.transformer(x)
        logits = self.head(x[:, 0])
        #如果传入真实标签，就直接计算损失值
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_classes), labels.view(-1))
            return loss
        else:
            return logits, attn_weights

import ml_collections
import argparse
import torch
import os
import numpy as np
from torch.utils.data import DataLoader

def get_config():
    '''
    配置transformer的模型的参数
    '''
    config = ml_collections.ConfigDict()
    config.patches = ml_collections.ConfigDict({'size':16})
    config.hidden_size = 768
    config.transformer = ml_collections.ConfigDict()
    config.transformer.mlp_dim = 3072
    config.transformer.num_heads = 12
    config.transformer.num_layers = 12
    config.transformer.attention_dropout_rate = 0.0
    config.transformer.dropout_rate = 0.1
    config.classifier = 'token'
    config.representation_size = None
    return config


def save_model(args, model,epoch_index):
    '''
    保存每个epoch训练的模型
    '''
    model_to_save = model.module if hasattr(model, 'module') else model
    model_checkpoint = os.path.join(args.output_dir, "epoch%s_checkpoint.bin" % epoch_index)
    torch.save(model_to_save.state_dict(), model_checkpoint)



#实例化模型
def getVisionTransformers_model(args):
    config=get_config()#获取模型的配置文件
    num_classes = 3
    model = VisionTransformer(config, args.img_size, zero_head=True, num_classes=num_classes)
    model.to(args.device)
    return args,model


#用测试集评估模型的训练好坏
def eval(args,model,test_loader):
    eval_loss=0.0
    total_acc=0.0
    model.eval()
    loss_function = torch.nn.CrossEntropyLoss()
    for i,batch in enumerate(test_loader):
        batch = tuple(t.to(args.device) for t in batch)
        x, y = batch
        with torch.no_grad():
            logits,_= model(x)#model返回的是（bs,num_classes）和weight
            batch_loss=loss_function(logits,y)
            #记录误差
            eval_loss+=batch_loss.item()
            #记录准确率
            _,preds= logits.max(1)
            num_correct=(preds==y).sum().item()
            total_acc+=num_correct

    loss=eval_loss/len(test_loader)
    acc=total_acc/(len(test_loader)*args.eval_batch_size)
    return loss,acc
from tqdm import tqdm
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    loss_function = torch.nn.CrossEntropyLoss()
    accu_loss = torch.zeros(1).to(device)  # 用于累计损失
    accu_num = torch.zeros(1).to(device)   # 用于累计预测正确的样本数
    optimizer.zero_grad()

    sample_num = 0
    data_loader = tqdm(data_loader)
    for step, data in enumerate(data_loader):  # step对应索引 data对应所传入的dataloader参数中的每个元素
        images, labels = data
        sample_num += images.shape[0]  # batchsize维度的值求和，即样本数量
        #前向
        pred = model(images.to(device))  # 预测结果
        pred_classes = torch.max(pred, dim=1)[1]
        # 在dim=1维度找到预测值最大的值，即为预测类；
        # torch.max()得到{max, max_indices}，使用torch[1]取出该张量的dim=1维度的向量，即max_indices向量（size=batchsize*类别数），为预测最大值对应的类别索引
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()
        # 判断max_indices与label（label是batchsize*类别数的向量）是否相等，相等返回True；并累计预测正确的样本数

        loss = loss_function(pred, labels.to(device))  # loss函数的输入参数是[pre，label]
        loss.backward()
        accu_loss += loss.detach()

        data_loader.desc = "[train epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step + 1),
                                                                               accu_num.item() / sample_num)

        if not torch.isfinite(loss):  # loss = inf or -inf or nan 的时候结束训练
            print('WARNING: non-finite loss, ending training ', loss)
            sys.exit(1)

        optimizer.step()
        optimizer.zero_grad()

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num
def evaluate(model, data_loader, device, epoch):
    loss_function = torch.nn.CrossEntropyLoss()

    model.eval()

    accu_num = torch.zeros(1).to(device)   # 累计预测正确的样本数
    accu_loss = torch.zeros(1).to(device)  # 累计损失

    sample_num = 0
    data_loader = tqdm(data_loader)
    for step, data in enumerate(data_loader):
        images, labels = data
        sample_num += images.shape[0]

        pred = model(images.to(device))
        pred_classes = torch.max(pred, dim=1)[1]
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()

        loss = loss_function(pred, labels.to(device))
        accu_loss += loss

        data_loader.desc = "[valid epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step + 1),
                                                                               accu_num.item() / sample_num)

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num
def train(args,model):
    device="cuda"
    print("load dataset.........................")
    #加载数据
    dataset=MyDataSet("train/","./Data/train.csv",data_transforms["train"],"train")
    indices = list(range(len(dataset)))
    shuffle_dataset = True
    random_seed= 42
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    val=MyDataSet("test/","./Data/test.csv",data_transforms["val"],"val")
    # 验证集比例
    indices = list(range(len(val)))
    random_seed= 48
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    train_loader = DataLoader(dataset, batch_size=8, shuffle=shuffle_dataset)
    val_loader=DataLoader(val, batch_size=8, shuffle=shuffle_dataset)
    # Prepare optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=args.learning_rate,
                                momentum=0.9,
                                weight_decay=args.weight_decay)

    print("training.........................")
    #设置测试损失list,和测试acc 列表
    val_loss_list=[]
    val_acc_list=[]
    #设置训练损失list
    train_loss_list=[]
    epoch =1
#从timm库导入模型vit_base_patch16_224_in21k
    model = create_model(num_classes=3).to("cuda")  
    epoch_number = 0
    for i in range(args.total_epoch):
        model.train()
        train_loss=0
        train_loss, train_acc = train_one_epoch(model=model,
                                                optimizer=optimizer,
                                                data_loader=train_loader,
                                                device=device,
                                                epoch=i)
        # validate
        val_loss, val_acc = evaluate(model=model,
                                     data_loader=val_loader,
                                     device=device,
                                     epoch=i)
        scheduler.step()
        print("train Epoch:{},loss:{},acc:{}".format(i,train_loss,train_acc))
        print("train Epoch:{},loss:{},acc:{}".format(i,val_loss,val_acc))
        # 每个epcoh保存一次模型参数
        save_model(args, model,epoch_number)
        # 每训练一个epoch,用当前训练的模型对验证集进行测试
        eval_loss, eval_acc = eval(args, model, val_loader)
        #将每一个测试集验证的结果加入列表
        """val_loss_list.append(eval_loss)
        val_acc_list.append(eval_acc)
        np.savetxt("val_loss_list(68).txt",val_loss_list)
        np.savetxt("val_acc_list(68).txt",val_acc_list)
        print("val Epoch:{},eval_loss:{},eval_acc:{}".format(epoch_number, eval_loss, eval_acc))"""
        epoch_number = epoch_number +1

def main():
    parser = argparse.ArgumentParser()
    # Required parameters
    """parser.add_argument("--dataset", choices=["cifar10", "cifar100"], default="cifar10",
                        help="Which downstream task.")"""
    parser.add_argument("--output_dir", default="./output", type=str,
                        help="The output directory where checkpoints will be written.")
    parser.add_argument("--img_size", default=224, type=int,help="Resolution size")
    parser.add_argument("--train_batch_size", default=8, type=int,
                        help="Total batch size for training.")
    parser.add_argument("--eval_batch_size", default=16, type=int,
                        help="Total batch size for eval.")
    parser.add_argument("--learning_rate", default=3e-2, type=float,
                        help="The initial learning rate for SGD.")
    parser.add_argument("--weight_decay", default=0, type=float,
                        help="Weight deay if we apply some.")
    parser.add_argument("--total_epoch", default=100, type=int,
                        help="Total number of training epochs to perform.")

    args = parser.parse_args(args=[])
    device = torch.device("cuda")
    args.device = device

    args,modle=getVisionTransformers_model(args)
    start=time.perf_counter()
    train(args,modle)
    end=time.perf_counter()
    print(end-start)

if __name__ == "__main__":
    main()



OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB (GPU 0; 31.75 GiB total capacity; 29.10 GiB already allocated; 16.50 MiB free; 30.53 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [58]:
import codecs
import os
import shutil
from PIL import Image
import csv
import torch
from torch.nn.modules import *
from functools import partial
import math
import time
"""all_file_dir='./Data'
train_file_dir = './Data/train'
test_file_dir='./Data/test'
val_file_dir='./Data/val'
class_list = [c for c in os.listdir(train_file_dir) if os.path.isdir(os.path.join(train_file_dir, c)) and not c.endswith('Set') and not c.startswith('.')]
class_list.sort()
print(class_list)
train_image_dir = os.path.join(all_file_dir, "trainImageSet")
if not os.path.exists(train_image_dir):
    os.makedirs(train_image_dir)
    
test_image_dir = os.path.join(all_file_dir, "testImageSet")
if not os.path.exists(test_image_dir):
    os.makedirs(test_image_dir)

val_image_dir = os.path.join(all_file_dir, "valImageSet")
if not os.path.exists(val_image_dir):
    os.makedirs(val_image_dir)
    
train_file = open("./Data/train.csv", 'w')
test_file = open(os.path.join(all_file_dir, "test.csv"), 'w')
val_file = open(os.path.join(all_file_dir, "val.csv"), 'w')

with codecs.open(os.path.join(all_file_dir, "label_list.csv"), "w") as label_list:
    #加入训练集
    label_id = 0
    for class_dir in class_list:
        csv.writer(label_list).writerow([label_id, class_dir])
        image_path_pre = os.path.join(train_file_dir, class_dir)
                # 存在一些文件打不开，此处需要稍作清洗
        label_id += 1
    label_id = 0
    for class_dir in class_list:
        print("1")
        image_path_pre = os.path.join(train_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(train_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(train_file).writerow([os.path.join(train_image_dir, file), label_id])
            except Exception as e:
                pass
    ##加入测试集
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(test_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(test_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(test_file).writerow([os.path.join(test_image_dir, file), label_id])
            except Exception as e:
                pass
                # 存在一些文件打不开，此处需要稍作清洗
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(val_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(val_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(val_file).writerow([os.path.join(val_image_dir, file), label_id])
            except Exception as e:
                pass
            
train_file.close()
test_file.close()  
val_file.close()  """

import torch
import pandas as pd
import numpy as np
import os
from PIL import Image
from torch.utils.data import Dataset
from torch import nn
from torch.nn import Conv2d
from torch.nn import Dropout
import copy 

class MyDataSet(Dataset):
    def __init__(self,image_path,csv_path,transforms,phrase):
        # 读取csv
        csv=pd.read_csv(csv_path,header=None)
        # 读取第一列,组合成完整的图片地址
        self.imgs=[str(k) for k in csv[0].values]
        self.phrase=phrase
        if self.phrase!="test":
            self.labels=np.asarray([k for k in csv[1].values])
        self.transforms=transforms
    

    def __getitem__(self, index):
        img_path = self.imgs[index]
        pil_img = Image.open(img_path).convert("RGB")
        if self.transforms:
            data = self.transforms(pil_img)
        else:
            pil_img = np.asarray(pil_img)
            data = torch.from_numpy(pil_img)
        if self.phrase!="test":
            label=self.labels[index]
            sample = (data,label)
        else:
            sample=data
        return sample

    def __len__(self):
        return len(self.imgs)
from torchvision import transforms
# 数据增强
data_transforms={
    "train":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "val":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "test":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    
}

class Embeddings(nn.Module):
    '''
    对图像进行编码，把图片当做一个句子，把图片分割成块，每一块表示一个单词
    '''
    def __init__(self,config,img_size,in_channels=3):
        super(Embeddings,self).__init__()
        img_size=img_size#224
        patch_size=config.patches["size"]#16
        ##将图片分割成多少块（224/16）*（224/16）=196
        n_patches=(img_size//patch_size)*(img_size//patch_size)
        #对图片进行卷积获取图片的块，并且将每一块映射成config.hidden_size维（768）
        self.patch_embeddings=Conv2d(in_channels=in_channels,
                                     out_channels=config.hidden_size,
                                     kernel_size=patch_size,
                                     stride=patch_size)
        
        #设置可学习的位置编码信息，（1,196+1,786）
        self.position_embeddings=nn.Parameter(torch.zeros(1,
                                                          n_patches+1,
                                                          config.hidden_size))
        #设置可学习的分类信息的维度
        self.classifer_token=nn.Parameter(torch.zeros(1,1,config.hidden_size))
        self.dropout=Dropout((config.transformer["dropout_rate"]))

    def forward(self,x):
        bs=x.shape[0]
        #cls_tokens=self.classifer_token.expand(bs,-1,-1)(bs,1,768)
        x=self.patch_embeddings(x)#（bs,768,14,14）
        x=x.flatten(2)#(bs,768,196)
        x=x.transpose(-1,-2)#(bs,196,768)
        cls_tokens=self.classifer_token.expand(bs,-1,-1)
        x=torch.cat((cls_tokens,x),dim=1)#将分类信息与图片块进行拼接（bs,197,768）
        embeddings=x+self.position_embeddings#将图片块信息和对其位置信息进行相加(bs,197,768)
        embeddings=self.dropout(embeddings)
        return  embeddings
# --------------------------------------- #
#（1）patch embedding
'''
img_size=224 : 输入图像的宽高
patch_size=16 ： 每个patch的宽高，也是卷积核的尺寸和步长
in_c=3 ： 输入图像的通道数
embed_dim=768 ： 卷积输出通道数
'''
# --------------------------------------- #
class patchembed(nn.Module):
    # 初始化
    def __init__(self, img_size=224, patch_size=16, in_c=3, embed_dim=768):
        super(patchembed, self).__init__()
        
        # 输入图像的尺寸224*224
        self.img_size = (img_size, img_size)
        # 每个patch的大小16*16
        self.patch_size = (patch_size, patch_size)
        # 将输入图像划分成14*14个patch
        self.grid_size = (img_size//patch_size, img_size//patch_size)
        # 一共有14*14个patch
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        
        # 使用16*16的卷积切分图像，将图像分成14*14个
        self.proj = nn.Conv2d(in_channels=in_c, out_channels=embed_dim, 
                              kernel_size=patch_size, stride=patch_size)
        
        # 定义标准化方法，给LN传入默认参数eps
        norm_layer = partial(nn.LayerNorm, eps=1e-6)
        self.norm = norm_layer(embed_dim)
        
        
    # 前向传播
    def forward(self, inputs):
        # 获得输入图像的shape
        B, C, H, W = inputs.shape
        
        # 如果输入图像的宽高不等于224*224就报错
        assert H==self.img_size[0] and W==self.img_size[1], 'input shape does not match 224*224'
        
        # 卷积层切分patch [b,3,224,224]==>[b,768,14,14]
        x = self.proj(inputs)
        # 展平 [b,768,14,14]==>[b,768,14*14]
        x = x.flatten(start_dim=2, end_dim=-1)  # 将索引为 start_dim 和 end_dim 之间（包括该位置）的数量相乘
        # 维度调整 [b,768,14*14]==>[b,14*14,768]
        x = x.transpose(1, 2)  # 实现一个张量的两个轴之间的维度转换
        # 标准化
        x = self.norm(x)
        
        return x

#2.构建self-Attention模块
class Attention(nn.Module):
    def __init__(self,config,vis):
        super(Attention,self).__init__()
        self.vis=vis
        self.num_attention_heads=config.transformer["num_heads"]#12
        self.attention_head_size = int(config.hidden_size / self.num_attention_heads)  # 768/12=64
        self.all_head_size = self.num_attention_heads * self.attention_head_size  # 12*64=768

        self.query = Linear(config.hidden_size, self.all_head_size)#wm,768->768，Wq矩阵为（768,768）
        self.key = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wk矩阵为（768,768）
        self.value = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wv矩阵为（768,768）
        self.out = Linear(config.hidden_size, config.hidden_size)  # wm,768->768
        self.attn_dropout = Dropout(config.transformer["attention_dropout_rate"])
        self.proj_dropout = Dropout(config.transformer["attention_dropout_rate"])

        self.softmax = Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (
        self.num_attention_heads, self.attention_head_size)  # wm,(bs,197)+(12,64)=(bs,197,12,64)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)  # wm,(bs,12,197,64)

    def forward(self, hidden_states):
        # hidden_states为：(bs,197,768)
        mixed_query_layer = self.query(hidden_states)#wm,768->768
        mixed_key_layer = self.key(hidden_states)#wm,768->768
        mixed_value_layer = self.value(hidden_states)#wm,768->768

        query_layer = self.transpose_for_scores(mixed_query_layer)#wm，(bs,12,197,64)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))#将q向量和k向量进行相乘（bs,12,197,197)
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)#将结果除以向量维数的开方
        attention_probs = self.softmax(attention_scores)#将得到的分数进行softmax,得到概率
        weights = attention_probs if self.vis else None#wm,实际上就是权重
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)#将概率与内容向量相乘
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)#wm,(bs,197)+(768,)=(bs,197,768)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights#wm,(bs,197,768),(bs,197,197)

#3.构建前向传播神经网络
#两个全连接神经网络，中间加了激活函数
class Mlp(nn.Module):
    def __init__(self, config):
        super(Mlp, self).__init__()
        self.fc1 = Linear(config.hidden_size, config.transformer["mlp_dim"])#wm,786->3072
        self.fc2 = Linear(config.transformer["mlp_dim"], config.hidden_size)#wm,3072->786
        self.act_fn = torch.nn.functional.gelu#wm,激活函数
        self.dropout = Dropout(config.transformer["dropout_rate"])

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.normal_(self.fc1.bias, std=1e-6)
        nn.init.normal_(self.fc2.bias, std=1e-6)

    def forward(self, x):
        x = self.fc1(x)#wm,786->3072
        x = self.act_fn(x)#激活函数
        x = self.dropout(x)#wm,丢弃
        x = self.fc2(x)#wm3072->786
        x = self.dropout(x)
        return x

#4.构建编码器的可重复利用的Block()模块：每一个block包含了self-attention模块和MLP模块
class Block(nn.Module):
    def __init__(self, config, vis):
        super(Block, self).__init__()
        self.hidden_size = config.hidden_size#wm,768
        self.attention_norm = LayerNorm(config.hidden_size, eps=1e-6)#wm，层归一化
        self.ffn_norm = LayerNorm(config.hidden_size, eps=1e-6)
        
        self.ffn = Mlp(config)
        self.attn = Attention(config, vis)

    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x = x + h#残差结构

        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x = x + h#残差结构
        return x, weights

#5.构建Encoder模块，该模块实际上就是堆叠N个Block模块
class Encoder(nn.Module):
    def __init__(self, config, vis):
        super(Encoder, self).__init__()
        self.vis = vis
        self.layer = nn.ModuleList()
        self.encoder_norm = LayerNorm(config.hidden_size, eps=1e-6)
        for _ in range(config.transformer["num_layers"]):
            layer = Block(config, vis)
            self.layer.append(copy.deepcopy(layer))

    def forward(self, hidden_states):
        attn_weights = []
        for layer_block in self.layer:
            hidden_states, weights = layer_block(hidden_states)
            if self.vis:
                attn_weights.append(weights)
        encoded = self.encoder_norm(hidden_states)
        return encoded, attn_weights

#6构建transformers完整结构，首先图片被embedding模块编码成序列数据，然后送入Encoder中进行编码
class Transformer(nn.Module):
    def __init__(self, config, img_size, vis):
        super(Transformer, self).__init__()
        self.embeddings = Embeddings(config, img_size=img_size)#wm,对一幅图片进行切块编码，得到的是（bs,n_patch+1（196）,每一块的维度（768））
        self.encoder = Encoder(config, vis)

    def forward(self, input_ids):
        embedding_output = self.embeddings(input_ids)#wm,输出的是（bs,196,768)
        encoded, attn_weights = self.encoder(embedding_output)#wm,输入的是（bs,196,768)
        return encoded, attn_weights#输出的是（bs,197,768）

#7构建VisionTransformer，用于图像分类
class VisionTransformer(nn.Module):
    def __init__(self, config, img_size=224, num_classes=21843, zero_head=False, vis=False):
        super(VisionTransformer, self).__init__()
        self.num_classes = num_classes
        self.zero_head = zero_head
        self.classifier = config.classifier

        self.transformer = Transformer(config, img_size, vis)
        self.head = Linear(config.hidden_size, num_classes)#wm,768-->10

    def forward(self, x, labels=None):
        x, attn_weights = self.transformer(x)
        print(x))
        logits = self.head(x[:, 0])
        print(attn_weights)
        #如果传入真实标签，就直接计算损失值
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_classes), labels.view(-1))
            return loss
        else:
            return logits, attn_weights

import ml_collections
import argparse
import torch
import os
import numpy as np
from torch.utils.data import DataLoader

def get_config():
    '''
    配置transformer的模型的参数
    '''
    config = ml_collections.ConfigDict()
    config.patches = ml_collections.ConfigDict({'size':16})
    config.hidden_size = 768
    config.transformer = ml_collections.ConfigDict()
    config.transformer.mlp_dim = 3072
    config.transformer.num_heads = 12
    config.transformer.num_layers = 12
    config.transformer.attention_dropout_rate = 0.0
    config.transformer.dropout_rate = 0.1
    config.classifier = 'token'
    config.representation_size = None
    return config


"""def save_model(args, model,epoch_index):
    '''
    保存每个epoch训练的模型
    '''
    model_to_save = model.module if hasattr(model, 'module') else model
    model_checkpoint = os.path.join(args.output_dir, "epoch%s_checkpoint.bin" % epoch_index)
    torch.save(model_to_save.state_dict(), model_checkpoint)"""



#实例化模型
def getVisionTransformers_model(args):
    config=get_config()#获取模型的配置文件
    num_classes = 3
    model = VisionTransformer(config, args.img_size, zero_head=True, num_classes=num_classes)
    model.to(args.device)
    return args,model


#用测试集评估模型的训练好坏
def eval(args,model,test_loader):
    eval_loss=0.0
    total_acc=0.0
    model.eval()
    loss_function = torch.nn.CrossEntropyLoss()
    for i,batch in enumerate(test_loader):
        batch = tuple(t.to(args.device) for t in batch)
        x, y = batch
        with torch.no_grad():
            logits,_= model(x)#model返回的是（bs,num_classes）和weight
            #记录准确率
            _,preds= logits.max(1)
            num_correct=(preds==y).sum().item()
            total_acc+=num_correct

    loss=eval_loss/len(test_loader)
    acc=total_acc/(len(test_loader)*args.eval_batch_size)
    return loss,acc


def train(args,model):
    print("load dataset.........................")
    #加载数据
    dataset=MyDataSet("train/","./Data/train.csv",data_transforms["train"],"train")
    indices = list(range(len(dataset)))
    shuffle_dataset = True
    random_seed= 42
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    val=MyDataSet("test/","./Data/test.csv",data_transforms["val"],"val")
    # 验证集比例
    indices = list(range(len(val)))
    random_seed= 42
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    train_loader = DataLoader(dataset, batch_size=8, shuffle=shuffle_dataset)
    val_loader=DataLoader(val, batch_size=16, shuffle=shuffle_dataset)
    # Prepare optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=args.learning_rate,
                                momentum=0.9,
                                weight_decay=args.weight_decay)

    print("training.........................")
    #设置测试损失list,和测试acc 列表
    val_loss_list=[]
    val_acc_list=[]
    #设置训练损失list
    train_loss_list=[]
    loaded_checkpoint = torch.load("./output/epoch66_checkpoint.bin") #载入原先保存的checkpoint参数
    epoch_number = 61
    model.load_state_dict(loaded_checkpoint) 
    eval_loss, eval_acc = eval(args, model, val_loader)
        #将每一个测试集验证的结果加入列表
    print("eval_acc:{}".format(eval_acc))

def main():
    parser = argparse.ArgumentParser()
    # Required parameters
    """parser.add_argument("--dataset", choices=["cifar10", "cifar100"], default="cifar10",
                        help="Which downstream task.")"""
    parser.add_argument("--output_dir", default="./output", type=str,
                        help="The output directory where checkpoints will be written.")
    parser.add_argument("--img_size", default=224, type=int,help="Resolution size")
    parser.add_argument("--train_batch_size", default=8, type=int,
                        help="Total batch size for training.")
    parser.add_argument("--eval_batch_size", default=16, type=int,
                        help="Total batch size for eval.")
    parser.add_argument("--learning_rate", default=3e-2, type=float,
                        help="The initial learning rate for SGD.")
    parser.add_argument("--weight_decay", default=0, type=float,
                        help="Weight deay if we apply some.")
    parser.add_argument("--total_epoch", default=100, type=int,
                        help="Total number of training epochs to perform.")

    args = parser.parse_args(args=[])
    device = torch.device("cuda")
    args.device = device

    args,modle=getVisionTransformers_model(args)
    start=time.perf_counter()
    train(args,modle)
    end=time.perf_counter()
    print(end-start)

if __name__ == "__main__":
    main()



IndentationError: unindent does not match any outer indentation level (<tokenize>, line 512)

In [1]:
!wget -P /home/work/data http://easydl-download.bj.bcebos.com/notebook-dataset/fileDownload && chmod +x /home/work/data/fileDownload && /home/work/data/fileDownload -token=G4juMkAyI2KnS46BELjaE68otTmEui7PsHctgVXLigsau+WnctPTsEXDWFjgb8OFCF3k5cEHq78XqnnH+epKWu9IP8XEIWVrg8uPO35oEb8D/kkw55i1+Fl1Z67/Z3z3ffY+Svs99i2Yq8v7tAQuwJ3CM2/utM+GtxvtoJ3Foz6kMX7DuFaD8Lln9uvyZ27WptWXX8yd1Vm1nv7jeY2sgYNndLv1zZu06MsK7BoZPuECCqljNsIU8LuVQMnyGSWccUdDyTz1Wul/r10aEsrAD6zJyJloROVB70FRfZbEtaxuOl/JbFoyWBEvMfKSMlF4Gzr0NKtnzEjcFwfMsn32LGlRGyf28d2qscoAm7Rm3f+PWVQ7Izrd/yTrYkNJkV++/ZtD0NFaqXnim8i8c4DpnQ==

--2023-02-20 22:16:46--  http://easydl-download.bj.bcebos.com/notebook-dataset/fileDownload
Resolving easydl-download.bj.bcebos.com (easydl-download.bj.bcebos.com)... 182.61.200.229, 182.61.200.195, 2409:8c04:1001:1002:0:ff:b001:368a
Connecting to easydl-download.bj.bcebos.com (easydl-download.bj.bcebos.com)|182.61.200.229|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6626563 (6.3M) [application/octet-stream]
Saving to: '/home/work/data/fileDownload'

fileDownload        100%[===================>]   6.32M  19.6MB/s    in 0.3s    

2023-02-20 22:16:47 (19.6 MB/s) - '/home/work/data/fileDownload' saved [6626563/6626563]

保存文件成功：/home/work/data/rc-upload-1676901552290-4-0000.epoch_smallvit_177_checkpoint.zip

In [2]:
!unzip data/rc-upload-1676901552290-4-0000.epoch_smallvit_177_checkpoint.zip

Archive:  data/rc-upload-1676901552290-4-0000.epoch_smallvit_177_checkpoint.zip
  inflating: epoch(smallvit)177_checkpoint.bin  


In [ ]:
import codecs
import os
import shutil
from PIL import Image
import ml_collections
import csv
import torch
from torch.nn.modules import *
from functools import partial
import math
import time
"""all_file_dir='./Data'
train_file_dir = './Data/train'
test_file_dir='./Data/test'
val_file_dir='./Data/val'
class_list = [c for c in os.listdir(train_file_dir) if os.path.isdir(os.path.join(train_file_dir, c)) and not c.endswith('Set') and not c.startswith('.')]
class_list.sort()
print(class_list)
train_image_dir = os.path.join(all_file_dir, "trainImageSet")
if not os.path.exists(train_image_dir):
    os.makedirs(train_image_dir)
    
test_image_dir = os.path.join(all_file_dir, "testImageSet")
if not os.path.exists(test_image_dir):
    os.makedirs(test_image_dir)

val_image_dir = os.path.join(all_file_dir, "valImageSet")
if not os.path.exists(val_image_dir):
    os.makedirs(val_image_dir)
    
train_file = open("./Data/train.csv", 'w')
test_file = open(os.path.join(all_file_dir, "test.csv"), 'w')
val_file = open(os.path.join(all_file_dir, "val.csv"), 'w')

with codecs.open(os.path.join(all_file_dir, "label_list.csv"), "w") as label_list:
    #加入训练集
    label_id = 0
    for class_dir in class_list:
        csv.writer(label_list).writerow([label_id, class_dir])
        image_path_pre = os.path.join(train_file_dir, class_dir)
                # 存在一些文件打不开，此处需要稍作清洗
        label_id += 1
    label_id = 0
    for class_dir in class_list:
        print("1")
        image_path_pre = os.path.join(train_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(train_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(train_file).writerow([os.path.join(train_image_dir, file), label_id])
            except Exception as e:
                pass
    ##加入测试集
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(test_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(test_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(test_file).writerow([os.path.join(test_image_dir, file), label_id])
            except Exception as e:
                pass
                # 存在一些文件打不开，此处需要稍作清洗
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(val_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(val_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(val_file).writerow([os.path.join(val_image_dir, file), label_id])
            except Exception as e:
                pass
            
train_file.close()
test_file.close()  
val_file.close()  """

import torch
import pandas as pd
import numpy as np
import os
from PIL import Image
from torch.utils.data import Dataset
from torch import nn
from torch.nn import Conv2d
from torch.nn import Dropout
import copy 

class MyDataSet(Dataset):
    def __init__(self,image_path,csv_path,transforms,phrase):
        # 读取csv
        csv=pd.read_csv(csv_path,header=None)
        # 读取第一列,组合成完整的图片地址
        self.imgs=[str(k) for k in csv[0].values]
        self.phrase=phrase
        if self.phrase!="test":
            self.labels=np.asarray([k for k in csv[1].values])
        self.transforms=transforms
    

    def __getitem__(self, index):
        img_path = self.imgs[index]
        pil_img = Image.open(img_path).convert("RGB")
        if self.transforms:
            data = self.transforms(pil_img)
        else:
            pil_img = np.asarray(pil_img)
            data = torch.from_numpy(pil_img)
        if self.phrase!="test":
            label=self.labels[index]
            sample = (data,label)
        else:
            sample=data
        return sample

    def __len__(self):
        return len(self.imgs)
from torchvision import transforms
# 数据增强
data_transforms={
    "train":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "val":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "test":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    
}

from math import sqrt
import torch
import torch.nn.functional as F
from torch import nn

from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# helpers

def pair(t):
    return t if isinstance(t, tuple) else (t, t)

# classes

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class LSA(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = dim_head *  heads
        self.heads = heads
        self.temperature = nn.Parameter(torch.log(torch.tensor(dim_head ** -0.5)))

        self.attend = nn.Softmax(dim = -1)
        self.dropout = nn.Dropout(dropout)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.temperature.exp()

        mask = torch.eye(dots.shape[-1], device = dots.device, dtype = torch.bool)
        mask_value = -torch.finfo(dots.dtype).max
        dots = dots.masked_fill(mask, mask_value)

        attn = self.attend(dots)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                PreNorm(dim, LSA(dim, heads = heads, dim_head = dim_head, dropout = dropout)),
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout = dropout))
            ]))
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x, attn(x)

class SPT(nn.Module):
    def __init__(self, *, dim, patch_size, channels = 3):
        super().__init__()
        patch_dim = patch_size * patch_size * 5 * channels

        self.to_patch_tokens = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_size, p2 = patch_size),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim)
        )

    def forward(self, x):
        shifts = ((1, -1, 0, 0), (-1, 1, 0, 0), (0, 0, 1, -1), (0, 0, -1, 1))
        shifted_x = list(map(lambda shift: F.pad(x, shift), shifts))
        x_with_shifts = torch.cat((x, *shifted_x), dim = 1)
        return self.to_patch_tokens(x_with_shifts)

class ViT(nn.Module):
    def __init__(self, *, image_size, patch_size=ml_collections.ConfigDict({'size':16}), num_classes=3, dim, depth=12, heads=12, mlp_dim=3072, pool = 'cls', channels = 3, dim_head = 64, dropout = 0., emb_dropout = 0.):
        super().__init__()
        image_height, image_width = pair(image_size)
        patch_height, patch_width = pair(patch_size)

        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'

        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width
        assert pool in {'cls', 'mean'}, 'pool type must be either cls (cls token) or mean (mean pooling)'

        self.to_patch_embedding = SPT(dim = dim, patch_size = patch_size, channels = channels)

        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.dropout = nn.Dropout(emb_dropout)

        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)

        self.pool = pool
        self.to_latent = nn.Identity()

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, img,labels = None):
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape

        cls_tokens = repeat(self.cls_token, '() n d -> b n d', b = b)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :(n + 1)]
        x = self.dropout(x)

        x, attn = self.transformer(x)
        x = x.mean(dim = 1) if self.pool == 'mean' else x[:, 0]

        x = self.to_latent(x)
        logits = self.mlp_head(x)
        loss_fct = CrossEntropyLoss()
        if labels is not None:
            loss = loss_fct(logits.view(-1, 3), labels.view(-1))
            return loss
        else:
            return logits, attn



import ml_collections
import argparse
import torch
import os
import numpy as np
from torch.utils.data import DataLoader

def get_config():
    '''
    配置transformer的模型的参数
    '''
    config = ml_collections.ConfigDict()
    config.patches = ml_collections.ConfigDict({'size':16})
    config.hidden_size = 768
    config.transformer = ml_collections.ConfigDict()
    config.transformer.mlp_dim = 3072
    config.transformer.num_heads = 12
    config.transformer.num_layers = 12
    config.transformer.attention_dropout_rate = 0.0
    config.transformer.dropout_rate = 0.1
    config.classifier = 'token'
    config.representation_size = None
    return config


def save_model(args, model,epoch_index):
    '''
    保存每个epoch训练的模型
    '''
    model_to_save = model.module if hasattr(model, 'module') else model
    model_checkpoint = os.path.join(args.output_dir, "epoch(smallvit)%s_checkpoint.bin" % epoch_index)
    torch.save(model_to_save.state_dict(), model_checkpoint)



#实例化模型
def getVisionTransformers_model(args):
    config=get_config()#获取模型的配置文件
    num_classes = 3
    model = ViT(image_size = 224,
    patch_size = 16,
    num_classes = 3,
    dim = 1024,
    depth = 6,
    heads = 16,
    mlp_dim = 2048,
    dropout = 0.1,
    emb_dropout = 0.1)
    model.to(args.device)
    return args,model


#用测试集评估模型的训练好坏
def eval(args,model,test_loader):
    eval_loss=0.0
    total_acc=0.0
    model.eval()
    loss_function = torch.nn.CrossEntropyLoss()
    for i,batch in enumerate(test_loader):
        batch = tuple(t.to(args.device) for t in batch)
        x, y = batch
        with torch.no_grad():
            logits,_= model(x)#model返回的是（bs,num_classes）和weight
            #记录准确率
            _,preds= logits.max(1)
            num_correct=(preds==y).sum().item()
            total_acc+=num_correct

    loss=eval_loss/len(test_loader)
    acc=total_acc/(len(test_loader)*args.eval_batch_size)
    return loss,acc


def train(args,model):
    print("load dataset.........................")
    #加载数据
    dataset=MyDataSet("train/","./Data/train.csv",data_transforms["train"],"train")
    indices = list(range(len(dataset)))
    shuffle_dataset = True
    random_seed= 42
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    val=MyDataSet("test/","./Data/test.csv",data_transforms["val"],"val")
    # 验证集比例
    indices = list(range(len(val)))
    random_seed= 42
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    train_loader = DataLoader(dataset, batch_size=8, shuffle=shuffle_dataset)
    val_loader=DataLoader(val, batch_size=16, shuffle=shuffle_dataset)
    # Prepare optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=args.learning_rate,
                                momentum=0.9,
                                weight_decay=args.weight_decay)

    print("training.........................")
    #设置测试损失list,和测试acc 列表
    val_loss_list=[]
    val_acc_list=[]
    #设置训练损失list
    train_loss_list=[]
    loaded_checkpoint = torch.load("epoch(smallvit)177_checkpoint.bin") #载入原先保存的checkpoint参数
    epoch_number = 178
    model.load_state_dict(loaded_checkpoint)
    for i in range(args.total_epoch):
        model.train()
        train_loss=0
        for step, batch in enumerate(train_loader):
            batch = tuple(t.to(args.device) for t in batch)
            x, y = batch
            loss = model(x, y)
            train_loss +=loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        #每训练一个epoch,记录一次训练损失
        train_loss=train_loss/len(train_loader)
        train_loss_list.append(train_loss)
        np.savetxt("train_loss_list.txt", train_loss_list)
        print("train Epoch:{},loss:{}".format(epoch_number,train_loss))

        # 每个epcoh保存一次模型参数
        save_model(args, model,epoch_number)
        # 每训练一个epoch,用当前训练的模型对验证集进行测试
        eval_loss, eval_acc = eval(args, model, val_loader)
        #将每一个测试集验证的结果加入列表
        val_loss_list.append(eval_loss)
        val_acc_list.append(eval_acc)
        np.savetxt("val_loss_list(smallvit-178).txt",val_loss_list)
        np.savetxt("val_acc_list(smallvit-178).txt",val_acc_list)
        print("val Epoch:{},eval_loss:{},eval_acc:{}".format(epoch_number, eval_loss, eval_acc))
        epoch_number=epoch_number+1

def main():
    parser = argparse.ArgumentParser()
    # Required parameters
    """parser.add_argument("--dataset", choices=["cifar10", "cifar100"], default="cifar10",
                        help="Which downstream task.")"""
    parser.add_argument("--output_dir", default="./output", type=str,
                        help="The output directory where checkpoints will be written.")
    parser.add_argument("--img_size", default=224, type=int,help="Resolution size")
    parser.add_argument("--train_batch_size", default=8, type=int,
                        help="Total batch size for training.")
    parser.add_argument("--eval_batch_size", default=16, type=int,
                        help="Total batch size for eval.")
    parser.add_argument("--learning_rate", default=3e-2, type=float,
                        help="The initial learning rate for SGD.")
    parser.add_argument("--weight_decay", default=0, type=float,
                        help="Weight deay if we apply some.")
    parser.add_argument("--total_epoch", default=100, type=int,
                        help="Total number of training epochs to perform.")

    args = parser.parse_args(args=[])
    device = torch.device("cuda")
    args.device = device

    args,modle=getVisionTransformers_model(args)
    start=time.perf_counter()
    train(args,modle)
    end=time.perf_counter()
    print(end-start)

if __name__ == "__main__":
    main()




load dataset.........................
training.........................
train Epoch:178,loss:0.49858856809102686
val Epoch:178,eval_loss:0.0,eval_acc:0.7323717948717948
train Epoch:179,loss:0.4956156184062636
val Epoch:179,eval_loss:0.0,eval_acc:0.7596153846153846
train Epoch:180,loss:0.4918263912932273
val Epoch:180,eval_loss:0.0,eval_acc:0.7387820512820513
train Epoch:181,loss:0.49248002600953256
val Epoch:181,eval_loss:0.0,eval_acc:0.7355769230769231
train Epoch:182,loss:0.4886970235480487
val Epoch:182,eval_loss:0.0,eval_acc:0.7131410256410257
train Epoch:183,loss:0.4933676341080044
val Epoch:183,eval_loss:0.0,eval_acc:0.7612179487179487
train Epoch:184,loss:0.49764870555122936
val Epoch:184,eval_loss:0.0,eval_acc:0.7467948717948718
train Epoch:185,loss:0.4977938602535637
val Epoch:185,eval_loss:0.0,eval_acc:0.7307692307692307
train Epoch:186,loss:0.48960151283378983
val Epoch:186,eval_loss:0.0,eval_acc:0.7115384615384616
train Epoch:187,loss:0.49086647965242525
val Epoch:187,eval_

In [1]:
import codecs
import os
import shutil
from PIL import Image
import csv
import torch
from torch.nn.modules import *
from functools import partial
import math
import time
import timm
"""all_file_dir='./Data'
train_file_dir = './Data/train'
test_file_dir='./Data/test'
val_file_dir='./Data/val'
class_list = [c for c in os.listdir(train_file_dir) if os.path.isdir(os.path.join(train_file_dir, c)) and not c.endswith('Set') and not c.startswith('.')]
class_list.sort()
print(class_list)
train_image_dir = os.path.join(all_file_dir, "trainImageSet")
if not os.path.exists(train_image_dir):
    os.makedirs(train_image_dir)
    
test_image_dir = os.path.join(all_file_dir, "testImageSet")
if not os.path.exists(test_image_dir):
    os.makedirs(test_image_dir)

val_image_dir = os.path.join(all_file_dir, "valImageSet")
if not os.path.exists(val_image_dir):
    os.makedirs(val_image_dir)
    
train_file = open("./Data/train.csv", 'w')
test_file = open(os.path.join(all_file_dir, "test.csv"), 'w')
val_file = open(os.path.join(all_file_dir, "val.csv"), 'w')

with codecs.open(os.path.join(all_file_dir, "label_list.csv"), "w") as label_list:
    #加入训练集
    label_id = 0
    for class_dir in class_list:
        csv.writer(label_list).writerow([label_id, class_dir])
        image_path_pre = os.path.join(train_file_dir, class_dir)
                # 存在一些文件打不开，此处需要稍作清洗
        label_id += 1
    label_id = 0
    for class_dir in class_list:
        print("1")
        image_path_pre = os.path.join(train_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(train_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(train_file).writerow([os.path.join(train_image_dir, file), label_id])
            except Exception as e:
                pass
    ##加入测试集
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(test_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(test_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(test_file).writerow([os.path.join(test_image_dir, file), label_id])
            except Exception as e:
                pass
                # 存在一些文件打不开，此处需要稍作清洗
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(val_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(val_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(val_file).writerow([os.path.join(val_image_dir, file), label_id])
            except Exception as e:
                pass
            
train_file.close()
test_file.close()  
val_file.close()  """
from timm.models.vision_transformer import vit_base_patch16_224_in21k as create_model  
import torch
import pandas as pd
import numpy as np
import os
from PIL import Image
from torch.utils.data import Dataset
from torch import nn
from torch.nn import Conv2d
from torch.nn import Dropout
import copy 

class MyDataSet(Dataset):
    def __init__(self,image_path,csv_path,transforms,phrase):
        # 读取csv
        csv=pd.read_csv(csv_path,header=None)
        # 读取第一列,组合成完整的图片地址
        self.imgs=[str(k) for k in csv[0].values]
        self.phrase=phrase
        if self.phrase!="test":
            self.labels=np.asarray([k for k in csv[1].values])
        self.transforms=transforms
    

    def __getitem__(self, index):
        img_path = self.imgs[index]
        pil_img = Image.open(img_path).convert("RGB")
        if self.transforms:
            data = self.transforms(pil_img)
        else:
            pil_img = np.asarray(pil_img)
            data = torch.from_numpy(pil_img)
        if self.phrase!="test":
            label=self.labels[index]
            sample = (data,label)
        else:
            sample=data
        return sample

    def __len__(self):
        return len(self.imgs)
from torchvision import transforms
# 数据增强
data_transforms={
    "train":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "val":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "test":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    
}

class Embeddings(nn.Module):
    '''
    对图像进行编码，把图片当做一个句子，把图片分割成块，每一块表示一个单词
    '''
    def __init__(self,config,img_size,in_channels=3):
        super(Embeddings,self).__init__()
        img_size=img_size#224
        patch_size=config.patches["size"]#16
        ##将图片分割成多少块（224/16）*（224/16）=196
        n_patches=(img_size//patch_size)*(img_size//patch_size)
        #对图片进行卷积获取图片的块，并且将每一块映射成config.hidden_size维（768）
        self.patch_embeddings=Conv2d(in_channels=in_channels,
                                     out_channels=config.hidden_size,
                                     kernel_size=patch_size,
                                     stride=patch_size)
        
        #设置可学习的位置编码信息，（1,196+1,786）
        self.position_embeddings=nn.Parameter(torch.zeros(1,
                                                          n_patches+1,
                                                          config.hidden_size))
        #设置可学习的分类信息的维度
        self.classifer_token=nn.Parameter(torch.zeros(1,1,config.hidden_size))
        self.dropout=Dropout((config.transformer["dropout_rate"]))

    def forward(self,x):
        bs=x.shape[0]
        #cls_tokens=self.classifer_token.expand(bs,-1,-1)(bs,1,768)
        x=self.patch_embeddings(x)#（bs,768,14,14）
        x=x.flatten(2)#(bs,768,196)
        x=x.transpose(-1,-2)#(bs,196,768)
        cls_tokens=self.classifer_token.expand(bs,-1,-1)
        x=torch.cat((cls_tokens,x),dim=1)#将分类信息与图片块进行拼接（bs,197,768）
        embeddings=x+self.position_embeddings#将图片块信息和对其位置信息进行相加(bs,197,768)
        embeddings=self.dropout(embeddings)
        return  embeddings
# --------------------------------------- #
#（1）patch embedding
'''
img_size=224 : 输入图像的宽高
patch_size=16 ： 每个patch的宽高，也是卷积核的尺寸和步长
in_c=3 ： 输入图像的通道数
embed_dim=768 ： 卷积输出通道数
'''
# --------------------------------------- #
class patchembed(nn.Module):
    # 初始化
    def __init__(self, img_size=224, patch_size=16, in_c=3, embed_dim=768):
        super(patchembed, self).__init__()
        
        # 输入图像的尺寸224*224
        self.img_size = (img_size, img_size)
        # 每个patch的大小16*16
        self.patch_size = (patch_size, patch_size)
        # 将输入图像划分成14*14个patch
        self.grid_size = (img_size//patch_size, img_size//patch_size)
        # 一共有14*14个patch
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        
        # 使用16*16的卷积切分图像，将图像分成14*14个
        self.proj = nn.Conv2d(in_channels=in_c, out_channels=embed_dim, 
                              kernel_size=patch_size, stride=patch_size)
        
        # 定义标准化方法，给LN传入默认参数eps
        norm_layer = partial(nn.LayerNorm, eps=1e-6)
        self.norm = norm_layer(embed_dim)
        
        
    # 前向传播
    def forward(self, inputs):
        # 获得输入图像的shape
        B, C, H, W = inputs.shape
        
        # 如果输入图像的宽高不等于224*224就报错
        assert H==self.img_size[0] and W==self.img_size[1], 'input shape does not match 224*224'
        
        # 卷积层切分patch [b,3,224,224]==>[b,768,14,14]
        x = self.proj(inputs)
        # 展平 [b,768,14,14]==>[b,768,14*14]
        x = x.flatten(start_dim=2, end_dim=-1)  # 将索引为 start_dim 和 end_dim 之间（包括该位置）的数量相乘
        # 维度调整 [b,768,14*14]==>[b,14*14,768]
        x = x.transpose(1, 2)  # 实现一个张量的两个轴之间的维度转换
        # 标准化
        x = self.norm(x)
        
        return x

#2.构建self-Attention模块
class Attention(nn.Module):
    def __init__(self,config,vis):
        super(Attention,self).__init__()
        self.vis=vis
        self.num_attention_heads=config.transformer["num_heads"]#12
        self.attention_head_size = int(config.hidden_size / self.num_attention_heads)  # 768/12=64
        self.all_head_size = self.num_attention_heads * self.attention_head_size  # 12*64=768

        self.query = Linear(config.hidden_size, self.all_head_size)#wm,768->768，Wq矩阵为（768,768）
        self.key = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wk矩阵为（768,768）
        self.value = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wv矩阵为（768,768）
        self.out = Linear(config.hidden_size, config.hidden_size)  # wm,768->768
        self.attn_dropout = Dropout(config.transformer["attention_dropout_rate"])
        self.proj_dropout = Dropout(config.transformer["attention_dropout_rate"])

        self.softmax = Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (
        self.num_attention_heads, self.attention_head_size)  # wm,(bs,197)+(12,64)=(bs,197,12,64)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)  # wm,(bs,12,197,64)

    def forward(self, hidden_states):
        # hidden_states为：(bs,197,768)
        mixed_query_layer = self.query(hidden_states)#wm,768->768
        mixed_key_layer = self.key(hidden_states)#wm,768->768
        mixed_value_layer = self.value(hidden_states)#wm,768->768

        query_layer = self.transpose_for_scores(mixed_query_layer)#wm，(bs,12,197,64)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))#将q向量和k向量进行相乘（bs,12,197,197)
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)#将结果除以向量维数的开方
        attention_probs = self.softmax(attention_scores)#将得到的分数进行softmax,得到概率
        weights = attention_probs if self.vis else None#wm,实际上就是权重
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)#将概率与内容向量相乘
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)#wm,(bs,197)+(768,)=(bs,197,768)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights#wm,(bs,197,768),(bs,197,197)

#3.构建前向传播神经网络
#两个全连接神经网络，中间加了激活函数
class Mlp(nn.Module):
    def __init__(self, config):
        super(Mlp, self).__init__()
        self.fc1 = Linear(config.hidden_size, config.transformer["mlp_dim"])#wm,786->3072
        self.fc2 = Linear(config.transformer["mlp_dim"], config.hidden_size)#wm,3072->786
        self.act_fn = torch.nn.functional.gelu#wm,激活函数
        self.dropout = Dropout(config.transformer["dropout_rate"])

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.normal_(self.fc1.bias, std=1e-6)
        nn.init.normal_(self.fc2.bias, std=1e-6)

    def forward(self, x):
        x = self.fc1(x)#wm,786->3072
        x = self.act_fn(x)#激活函数
        x = self.dropout(x)#wm,丢弃
        x = self.fc2(x)#wm3072->786
        x = self.dropout(x)
        return x

#4.构建编码器的可重复利用的Block()模块：每一个block包含了self-attention模块和MLP模块
class Block(nn.Module):
    def __init__(self, config, vis):
        super(Block, self).__init__()
        self.hidden_size = config.hidden_size#wm,768
        self.attention_norm = LayerNorm(config.hidden_size, eps=1e-6)#wm，层归一化
        self.ffn_norm = LayerNorm(config.hidden_size, eps=1e-6)
        
        self.ffn = Mlp(config)
        self.attn = Attention(config, vis)

    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x = x + h#残差结构

        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x = x + h#残差结构
        return x, weights

#5.构建Encoder模块，该模块实际上就是堆叠N个Block模块
class Encoder(nn.Module):
    def __init__(self, config, vis):
        super(Encoder, self).__init__()
        self.vis = vis
        self.layer = nn.ModuleList()
        self.encoder_norm = LayerNorm(config.hidden_size, eps=1e-6)
        for _ in range(config.transformer["num_layers"]):
            layer = Block(config, vis)
            self.layer.append(copy.deepcopy(layer))

    def forward(self, hidden_states):
        attn_weights = []
        for layer_block in self.layer:
            hidden_states, weights = layer_block(hidden_states)
            if self.vis:
                attn_weights.append(weights)
        encoded = self.encoder_norm(hidden_states)
        return encoded, attn_weights

#6构建transformers完整结构，首先图片被embedding模块编码成序列数据，然后送入Encoder中进行编码
class Transformer(nn.Module):
    def __init__(self, config, img_size, vis):
        super(Transformer, self).__init__()
        self.embeddings = Embeddings(config, img_size=img_size)#wm,对一幅图片进行切块编码，得到的是（bs,n_patch+1（196）,每一块的维度（768））
        self.encoder = Encoder(config, vis)

    def forward(self, input_ids):
        embedding_output = self.embeddings(input_ids)#wm,输出的是（bs,196,768)
        encoded, attn_weights = self.encoder(embedding_output)#wm,输入的是（bs,196,768)
        return encoded, attn_weights#输出的是（bs,197,768）

#7构建VisionTransformer，用于图像分类
class VisionTransformer(nn.Module):
    def __init__(self, config, img_size=224, num_classes=21843, zero_head=False, vis=False):
        super(VisionTransformer, self).__init__()
        self.num_classes = num_classes
        self.zero_head = zero_head
        self.classifier = config.classifier

        self.transformer = Transformer(config, img_size, vis)
        self.head = Linear(config.hidden_size, num_classes)#wm,768-->10

    def forward(self, x, labels=None):
        x, attn_weights = self.transformer(x)
        logits = self.head(x[:, 0])
        #如果传入真实标签，就直接计算损失值
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_classes), labels.view(-1))
            return loss
        else:
            return logits, attn_weights

import ml_collections
import argparse
import torch
import os
import numpy as np
from torch.utils.data import DataLoader

def get_config():
    '''
    配置transformer的模型的参数
    '''
    config = ml_collections.ConfigDict()
    config.patches = ml_collections.ConfigDict({'size':16})
    config.hidden_size = 768
    config.transformer = ml_collections.ConfigDict()
    config.transformer.mlp_dim = 3072
    config.transformer.num_heads = 12
    config.transformer.num_layers = 12
    config.transformer.attention_dropout_rate = 0.0
    config.transformer.dropout_rate = 0.1
    config.classifier = 'token'
    config.representation_size = None
    return config

def save_model(args, model,epoch_index):
    '''
    保存每个epoch训练的模型
    '''
    model_to_save = model.module if hasattr(model, 'module') else model
    model_checkpoint = os.path.join(args.output_dir, "epoch%s_checkpoint.bin" % epoch_index)
    torch.save(model_to_save.state_dict(), model_checkpoint)


#实例化模型
def getVisionTransformers_model(args):
    config=get_config()#获取模型的配置文件
    num_classes = 3
    model = VisionTransformer(config, args.img_size, zero_head=True, num_classes=num_classes)
    model.to(args.device)
    return args,model

#用测试集评估模型的训练好坏
def eval(args,model,test_loader):
    eval_loss=0.0
    total_acc=0.0
    model.eval()
    loss_function = torch.nn.CrossEntropyLoss()
    for i,batch in enumerate(test_loader):
        batch = tuple(t.to(args.device) for t in batch)
        x, y = batch
        with torch.no_grad():
            logits,_= model(x)#model返回的是（bs,num_classes）和weight
            batch_loss=loss_function(logits,y)
            #记录误差
            eval_loss+=batch_loss.item()
            #记录准确率
            _,preds= logits.max(1)
            num_correct=(preds==y).sum().item()
            total_acc+=num_correct

    loss=eval_loss/len(test_loader)
    acc=total_acc/(len(test_loader)*args.eval_batch_size)
    return loss,acc
from tqdm import tqdm
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    loss_function = torch.nn.CrossEntropyLoss()
    accu_loss = torch.zeros(1).to(device)  # 用于累计损失
    accu_num = torch.zeros(1).to(device)   # 用于累计预测正确的样本数
    optimizer.zero_grad()

    sample_num = 0
    data_loader = tqdm(data_loader)
    for step, data in enumerate(data_loader):  # step对应索引 data对应所传入的dataloader参数中的每个元素
        images, labels = data
        sample_num += images.shape[0]  # batchsize维度的值求和，即样本数量
        #前向
        pred = model(images.to(device))  # 预测结果
        pred_classes = torch.max(pred, dim=1)[1]
        # 在dim=1维度找到预测值最大的值，即为预测类；
        # torch.max()得到{max, max_indices}，使用torch[1]取出该张量的dim=1维度的向量，即max_indices向量（size=batchsize*类别数），为预测最大值对应的类别索引
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()
        # 判断max_indices与label（label是batchsize*类别数的向量）是否相等，相等返回True；并累计预测正确的样本数

        loss = loss_function(pred, labels.to(device))  # loss函数的输入参数是[pre，label]
        loss.backward()
        accu_loss += loss.detach()

        data_loader.desc = "[train epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step + 1),
                                                                               accu_num.item() / sample_num)

        if not torch.isfinite(loss):  # loss = inf or -inf or nan 的时候结束训练
            print('WARNING: non-finite loss, ending training ', loss)
            sys.exit(1)

        optimizer.step()
        optimizer.zero_grad()

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num
def evaluate(model, data_loader, device, epoch):
    loss_function = torch.nn.CrossEntropyLoss()

    model.eval()

    accu_num = torch.zeros(1).to(device)   # 累计预测正确的样本数
    accu_loss = torch.zeros(1).to(device)  # 累计损失

    sample_num = 0
    data_loader = tqdm(data_loader)
    for step, data in enumerate(data_loader):
        images, labels = data
        sample_num += images.shape[0]

        pred = model(images.to(device))
        pred_classes = torch.max(pred, dim=1)[1]
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()

        loss = loss_function(pred, labels.to(device))
        accu_loss += loss

        data_loader.desc = "[valid epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step + 1),
                                                                               accu_num.item() / sample_num)

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num
def train(args,model):
    device="cuda"
    print("load dataset.........................")
    #加载数据
    dataset=MyDataSet("train/","./Data/train.csv",data_transforms["train"],"train")
    indices = list(range(len(dataset)))
    shuffle_dataset = True
    random_seed= 42
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    val=MyDataSet("test/","./Data/test.csv",data_transforms["val"],"val")
    # 验证集比例
    indices = list(range(len(val)))
    random_seed= 48
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    train_loader = DataLoader(dataset, batch_size=8, shuffle=shuffle_dataset)
    val_loader=DataLoader(val, batch_size=8, shuffle=shuffle_dataset)
    # Prepare optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=args.learning_rate,
                                momentum=0.9,
                                weight_decay=args.weight_decay)

    print("training.........................")
    #设置测试损失list,和测试acc 列表
    val_loss_list=[]
    val_acc_list=[]
    #设置训练损失list
    train_loss_list=[]
    epoch =1
#从timm库导入模型vit_base_patch16_224_in21k
    model = create_model(num_classes=3).to("cuda")  
    epoch_number = 0
    for i in range(args.total_epoch):
        model.train()
        train_loss=0
        train_loss, train_acc = train_one_epoch(model=model,
                                                optimizer=optimizer,
                                                data_loader=train_loader,
                                                device=device,
                                                epoch=i)
        # validate
        val_loss, val_acc = evaluate(model=model,
                                     data_loader=val_loader,
                                     device=device,
                                     epoch=i)
        scheduler.step()
        print("train Epoch:{},loss:{},acc:{}".format(i,train_loss,train_acc))
        print("train Epoch:{},loss:{},acc:{}".format(i,val_loss,val_acc))
        # 每个epcoh保存一次模型参数
        save_model(args, model,epoch_number)
        # 每训练一个epoch,用当前训练的模型对验证集进行测试
        eval_loss, eval_acc = eval(args, model, val_loader)
        #将每一个测试集验证的结果加入列表
        """val_loss_list.append(eval_loss)
        val_acc_list.append(eval_acc)
        np.savetxt("val_loss_list(68).txt",val_loss_list)
        np.savetxt("val_acc_list(68).txt",val_acc_list)
        print("val Epoch:{},eval_loss:{},eval_acc:{}".format(epoch_number, eval_loss, eval_acc))"""
        epoch_number = epoch_number +1

def main():
    parser = argparse.ArgumentParser()
    # Required parameters
    """parser.add_argument("--dataset", choices=["cifar10", "cifar100"], default="cifar10",
                        help="Which downstream task.")"""
    parser.add_argument("--output_dir", default="./output", type=str,
                        help="The output directory where checkpoints will be written.")
    parser.add_argument("--img_size", default=224, type=int,help="Resolution size")
    parser.add_argument("--train_batch_size", default=8, type=int,
                        help="Total batch size for training.")
    parser.add_argument("--eval_batch_size", default=16, type=int,
                        help="Total batch size for eval.")
    parser.add_argument("--learning_rate", default=3e-2, type=float,
                        help="The initial learning rate for SGD.")
    parser.add_argument("--weight_decay", default=0, type=float,
                        help="Weight deay if we apply some.")
    parser.add_argument("--total_epoch", default=100, type=int,
                        help="Total number of training epochs to perform.")

    args = parser.parse_args(args=[])
    device = torch.device("cuda")
    args.device = device

    args,modle=getVisionTransformers_model(args)
    start=time.perf_counter()
    train(args,modle)
    end=time.perf_counter()
    print(end-start)

if __name__ == "__main__":
    main()




ModuleNotFoundError: No module named 'timm'

In [4]:
import codecs
import os
import shutil
from PIL import Image
import csv
import torch
from torch.nn.modules import *
from functools import partial
import math
import time
import timm
"""all_file_dir='./Data'
train_file_dir = './Data/train'
test_file_dir='./Data/test'
val_file_dir='./Data/val'
class_list = [c for c in os.listdir(train_file_dir) if os.path.isdir(os.path.join(train_file_dir, c)) and not c.endswith('Set') and not c.startswith('.')]
class_list.sort()
print(class_list)
train_image_dir = os.path.join(all_file_dir, "trainImageSet")
if not os.path.exists(train_image_dir):
    os.makedirs(train_image_dir)
    
test_image_dir = os.path.join(all_file_dir, "testImageSet")
if not os.path.exists(test_image_dir):
    os.makedirs(test_image_dir)

val_image_dir = os.path.join(all_file_dir, "valImageSet")
if not os.path.exists(val_image_dir):
    os.makedirs(val_image_dir)
    
train_file = open("./Data/train.csv", 'w')
test_file = open(os.path.join(all_file_dir, "test.csv"), 'w')
val_file = open(os.path.join(all_file_dir, "val.csv"), 'w')

with codecs.open(os.path.join(all_file_dir, "label_list.csv"), "w") as label_list:
    #加入训练集
    label_id = 0
    for class_dir in class_list:
        csv.writer(label_list).writerow([label_id, class_dir])
        image_path_pre = os.path.join(train_file_dir, class_dir)
                # 存在一些文件打不开，此处需要稍作清洗
        label_id += 1
    label_id = 0
    for class_dir in class_list:
        print("1")
        image_path_pre = os.path.join(train_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(train_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(train_file).writerow([os.path.join(train_image_dir, file), label_id])
            except Exception as e:
                pass
    ##加入测试集
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(test_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(test_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(test_file).writerow([os.path.join(test_image_dir, file), label_id])
            except Exception as e:
                pass
                # 存在一些文件打不开，此处需要稍作清洗
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(val_file_dir, class_dir)
        for file in os.listdir(image_path_pre):
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file), os.path.join(val_image_dir, file))
                if "bacteria" in file:
                    label_id = 1
                if "virus" in file:
                    label_id = 2
                csv.writer(val_file).writerow([os.path.join(val_image_dir, file), label_id])
            except Exception as e:
                pass
            
train_file.close()
test_file.close()  
val_file.close()  """
from timm.models.vision_transformer import vit_base_patch16_224_in21k as create_model  
import torch
import pandas as pd
import numpy as np
import os
from PIL import Image
from torch.utils.data import Dataset
from torch import nn
from torch.nn import Conv2d
from torch.nn import Dropout
import copy 

class MyDataSet(Dataset):
    def __init__(self,image_path,csv_path,transforms,phrase):
        # 读取csv
        csv=pd.read_csv(csv_path,header=None)
        # 读取第一列,组合成完整的图片地址
        self.imgs=[str(k) for k in csv[0].values]
        self.phrase=phrase
        if self.phrase!="test":
            self.labels=np.asarray([k for k in csv[1].values])
        self.transforms=transforms
    

    def __getitem__(self, index):
        img_path = self.imgs[index]
        pil_img = Image.open(img_path).convert("RGB")
        if self.transforms:
            data = self.transforms(pil_img)
        else:
            pil_img = np.asarray(pil_img)
            data = torch.from_numpy(pil_img)
        if self.phrase!="test":
            label=self.labels[index]
            sample = (data,label)
        else:
            sample=data
        return sample

    def __len__(self):
        return len(self.imgs)
from torchvision import transforms
# 数据增强
data_transforms={
    "train":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "val":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "test":
    transforms.Compose([
            transforms.Resize(size=(240,240)),
            transforms.CenterCrop(size=(224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    
}

class Embeddings(nn.Module):
    '''
    对图像进行编码，把图片当做一个句子，把图片分割成块，每一块表示一个单词
    '''
    def __init__(self,config,img_size,in_channels=3):
        super(Embeddings,self).__init__()
        img_size=img_size#224
        patch_size=config.patches["size"]#16
        ##将图片分割成多少块（224/16）*（224/16）=196
        n_patches=(img_size//patch_size)*(img_size//patch_size)
        #对图片进行卷积获取图片的块，并且将每一块映射成config.hidden_size维（768）
        self.patch_embeddings=Conv2d(in_channels=in_channels,
                                     out_channels=config.hidden_size,
                                     kernel_size=patch_size,
                                     stride=patch_size)
        
        #设置可学习的位置编码信息，（1,196+1,786）
        self.position_embeddings=nn.Parameter(torch.zeros(1,
                                                          n_patches+1,
                                                          config.hidden_size))
        #设置可学习的分类信息的维度
        self.classifer_token=nn.Parameter(torch.zeros(1,1,config.hidden_size))
        self.dropout=Dropout((config.transformer["dropout_rate"]))

    def forward(self,x):
        bs=x.shape[0]
        #cls_tokens=self.classifer_token.expand(bs,-1,-1)(bs,1,768)
        x=self.patch_embeddings(x)#（bs,768,14,14）
        x=x.flatten(2)#(bs,768,196)
        x=x.transpose(-1,-2)#(bs,196,768)
        cls_tokens=self.classifer_token.expand(bs,-1,-1)
        x=torch.cat((cls_tokens,x),dim=1)#将分类信息与图片块进行拼接（bs,197,768）
        embeddings=x+self.position_embeddings#将图片块信息和对其位置信息进行相加(bs,197,768)
        embeddings=self.dropout(embeddings)
        return  embeddings
# --------------------------------------- #
#（1）patch embedding
'''
img_size=224 : 输入图像的宽高
patch_size=16 ： 每个patch的宽高，也是卷积核的尺寸和步长
in_c=3 ： 输入图像的通道数
embed_dim=768 ： 卷积输出通道数
'''
# --------------------------------------- #
class patchembed(nn.Module):
    # 初始化
    def __init__(self, img_size=224, patch_size=16, in_c=3, embed_dim=768):
        super(patchembed, self).__init__()
        
        # 输入图像的尺寸224*224
        self.img_size = (img_size, img_size)
        # 每个patch的大小16*16
        self.patch_size = (patch_size, patch_size)
        # 将输入图像划分成14*14个patch
        self.grid_size = (img_size//patch_size, img_size//patch_size)
        # 一共有14*14个patch
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        
        # 使用16*16的卷积切分图像，将图像分成14*14个
        self.proj = nn.Conv2d(in_channels=in_c, out_channels=embed_dim, 
                              kernel_size=patch_size, stride=patch_size)
        
        # 定义标准化方法，给LN传入默认参数eps
        norm_layer = partial(nn.LayerNorm, eps=1e-6)
        self.norm = norm_layer(embed_dim)
        
        
    # 前向传播
    def forward(self, inputs):
        # 获得输入图像的shape
        B, C, H, W = inputs.shape
        
        # 如果输入图像的宽高不等于224*224就报错
        assert H==self.img_size[0] and W==self.img_size[1], 'input shape does not match 224*224'
        
        # 卷积层切分patch [b,3,224,224]==>[b,768,14,14]
        x = self.proj(inputs)
        # 展平 [b,768,14,14]==>[b,768,14*14]
        x = x.flatten(start_dim=2, end_dim=-1)  # 将索引为 start_dim 和 end_dim 之间（包括该位置）的数量相乘
        # 维度调整 [b,768,14*14]==>[b,14*14,768]
        x = x.transpose(1, 2)  # 实现一个张量的两个轴之间的维度转换
        # 标准化
        x = self.norm(x)
        
        return x

#2.构建self-Attention模块
class Attention(nn.Module):
    def __init__(self,config,vis):
        super(Attention,self).__init__()
        self.vis=vis
        self.num_attention_heads=config.transformer["num_heads"]#12
        self.attention_head_size = int(config.hidden_size / self.num_attention_heads)  # 768/12=64
        self.all_head_size = self.num_attention_heads * self.attention_head_size  # 12*64=768

        self.query = Linear(config.hidden_size, self.all_head_size)#wm,768->768，Wq矩阵为（768,768）
        self.key = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wk矩阵为（768,768）
        self.value = Linear(config.hidden_size, self.all_head_size)#wm,768->768,Wv矩阵为（768,768）
        self.out = Linear(config.hidden_size, config.hidden_size)  # wm,768->768
        self.attn_dropout = Dropout(config.transformer["attention_dropout_rate"])
        self.proj_dropout = Dropout(config.transformer["attention_dropout_rate"])

        self.softmax = Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (
        self.num_attention_heads, self.attention_head_size)  # wm,(bs,197)+(12,64)=(bs,197,12,64)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)  # wm,(bs,12,197,64)

    def forward(self, hidden_states):
        # hidden_states为：(bs,197,768)
        mixed_query_layer = self.query(hidden_states)#wm,768->768
        mixed_key_layer = self.key(hidden_states)#wm,768->768
        mixed_value_layer = self.value(hidden_states)#wm,768->768

        query_layer = self.transpose_for_scores(mixed_query_layer)#wm，(bs,12,197,64)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))#将q向量和k向量进行相乘（bs,12,197,197)
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)#将结果除以向量维数的开方
        attention_probs = self.softmax(attention_scores)#将得到的分数进行softmax,得到概率
        weights = attention_probs if self.vis else None#wm,实际上就是权重
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)#将概率与内容向量相乘
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)#wm,(bs,197)+(768,)=(bs,197,768)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights#wm,(bs,197,768),(bs,197,197)

#3.构建前向传播神经网络
#两个全连接神经网络，中间加了激活函数
class Mlp(nn.Module):
    def __init__(self, config):
        super(Mlp, self).__init__()
        self.fc1 = Linear(config.hidden_size, config.transformer["mlp_dim"])#wm,786->3072
        self.fc2 = Linear(config.transformer["mlp_dim"], config.hidden_size)#wm,3072->786
        self.act_fn = torch.nn.functional.gelu#wm,激活函数
        self.dropout = Dropout(config.transformer["dropout_rate"])

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.normal_(self.fc1.bias, std=1e-6)
        nn.init.normal_(self.fc2.bias, std=1e-6)

    def forward(self, x):
        x = self.fc1(x)#wm,786->3072
        x = self.act_fn(x)#激活函数
        x = self.dropout(x)#wm,丢弃
        x = self.fc2(x)#wm3072->786
        x = self.dropout(x)
        return x

#4.构建编码器的可重复利用的Block()模块：每一个block包含了self-attention模块和MLP模块
class Block(nn.Module):
    def __init__(self, config, vis):
        super(Block, self).__init__()
        self.hidden_size = config.hidden_size#wm,768
        self.attention_norm = LayerNorm(config.hidden_size, eps=1e-6)#wm，层归一化
        self.ffn_norm = LayerNorm(config.hidden_size, eps=1e-6)
        
        self.ffn = Mlp(config)
        self.attn = Attention(config, vis)

    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x = x + h#残差结构

        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x = x + h#残差结构
        return x, weights

#5.构建Encoder模块，该模块实际上就是堆叠N个Block模块
class Encoder(nn.Module):
    def __init__(self, config, vis):
        super(Encoder, self).__init__()
        self.vis = vis
        self.layer = nn.ModuleList()
        self.encoder_norm = LayerNorm(config.hidden_size, eps=1e-6)
        for _ in range(config.transformer["num_layers"]):
            layer = Block(config, vis)
            self.layer.append(copy.deepcopy(layer))

    def forward(self, hidden_states):
        attn_weights = []
        for layer_block in self.layer:
            hidden_states, weights = layer_block(hidden_states)
            if self.vis:
                attn_weights.append(weights)
        encoded = self.encoder_norm(hidden_states)
        return encoded, attn_weights

#6构建transformers完整结构，首先图片被embedding模块编码成序列数据，然后送入Encoder中进行编码
class Transformer(nn.Module):
    def __init__(self, config, img_size, vis):
        super(Transformer, self).__init__()
        self.embeddings = Embeddings(config, img_size=img_size)#wm,对一幅图片进行切块编码，得到的是（bs,n_patch+1（196）,每一块的维度（768））
        self.encoder = Encoder(config, vis)

    def forward(self, input_ids):
        embedding_output = self.embeddings(input_ids)#wm,输出的是（bs,196,768)
        encoded, attn_weights = self.encoder(embedding_output)#wm,输入的是（bs,196,768)
        return encoded, attn_weights#输出的是（bs,197,768）

#7构建VisionTransformer，用于图像分类
class VisionTransformer(nn.Module):
    def __init__(self, config, img_size=224, num_classes=21843, zero_head=False, vis=False):
        super(VisionTransformer, self).__init__()
        self.num_classes = num_classes
        self.zero_head = zero_head
        self.classifier = config.classifier

        self.transformer = Transformer(config, img_size, vis)
        self.head = Linear(config.hidden_size, num_classes)#wm,768-->10

    def forward(self, x, labels=None):
        x, attn_weights = self.transformer(x)
        logits = self.head(x[:, 0])
        #如果传入真实标签，就直接计算损失值
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_classes), labels.view(-1))
            return loss
        else:
            return logits, attn_weights

import ml_collections
import argparse
import torch
import os
import numpy as np
from torch.utils.data import DataLoader

def get_config():
    '''
    配置transformer的模型的参数
    '''
    config = ml_collections.ConfigDict()
    config.patches = ml_collections.ConfigDict({'size':16})
    config.hidden_size = 768
    config.transformer = ml_collections.ConfigDict()
    config.transformer.mlp_dim = 3072
    config.transformer.num_heads = 12
    config.transformer.num_layers = 12
    config.transformer.attention_dropout_rate = 0.0
    config.transformer.dropout_rate = 0.1
    config.classifier = 'token'
    config.representation_size = None
    return config

def save_model(args, model,epoch_index):
    '''
    保存每个epoch训练的模型
    '''
    model_to_save = model.module if hasattr(model, 'module') else model
    model_checkpoint = os.path.join(args.output_dir, "epoch%s_checkpoint.bin" % epoch_index)
    torch.save(model_to_save.state_dict(), model_checkpoint)


#实例化模型
def getVisionTransformers_model(args):
    config=get_config()#获取模型的配置文件
    num_classes = 3
    model = VisionTransformer(config, args.img_size, zero_head=True, num_classes=num_classes)
    model.to(args.device)
    return args,model

#用测试集评估模型的训练好坏
def eval(args,model,test_loader):
    eval_loss=0.0
    total_acc=0.0
    model.eval()
    loss_function = torch.nn.CrossEntropyLoss()
    for i,batch in enumerate(test_loader):
        batch = tuple(t.to(args.device) for t in batch)
        x, y = batch
        with torch.no_grad():
            logits,_= model(x)#model返回的是（bs,num_classes）和weight
            batch_loss=loss_function(logits,y)
            #记录误差
            eval_loss+=batch_loss.item()
            #记录准确率
            _,preds= logits.max(1)
            num_correct=(preds==y).sum().item()
            total_acc+=num_correct

    loss=eval_loss/len(test_loader)
    acc=total_acc/(len(test_loader)*args.eval_batch_size)
    return loss,acc
from tqdm import tqdm
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    loss_function = torch.nn.CrossEntropyLoss()
    accu_loss = torch.zeros(1).to(device)  # 用于累计损失
    accu_num = torch.zeros(1).to(device)   # 用于累计预测正确的样本数
    optimizer.zero_grad()

    sample_num = 0
    data_loader = tqdm(data_loader)
    for step, data in enumerate(data_loader):  # step对应索引 data对应所传入的dataloader参数中的每个元素
        images, labels = data
        sample_num += images.shape[0]  # batchsize维度的值求和，即样本数量
        #前向
        pred = model(images.to(device))  # 预测结果
        pred_classes = torch.max(pred, dim=1)[1]
        # 在dim=1维度找到预测值最大的值，即为预测类；
        # torch.max()得到{max, max_indices}，使用torch[1]取出该张量的dim=1维度的向量，即max_indices向量（size=batchsize*类别数），为预测最大值对应的类别索引
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()
        # 判断max_indices与label（label是batchsize*类别数的向量）是否相等，相等返回True；并累计预测正确的样本数

        loss = loss_function(pred, labels.to(device))  # loss函数的输入参数是[pre，label]
        loss.backward()
        accu_loss += loss.detach()

        data_loader.desc = "[train epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step + 1),
                                                                               accu_num.item() / sample_num)

        if not torch.isfinite(loss):  # loss = inf or -inf or nan 的时候结束训练
            print('WARNING: non-finite loss, ending training ', loss)
            sys.exit(1)

        optimizer.step()
        optimizer.zero_grad()

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num
def evaluate(model, data_loader, device, epoch):
    loss_function = torch.nn.CrossEntropyLoss()

    model.eval()

    accu_num = torch.zeros(1).to(device)   # 累计预测正确的样本数
    accu_loss = torch.zeros(1).to(device)  # 累计损失

    sample_num = 0
    data_loader = tqdm(data_loader)
    for step, data in enumerate(data_loader):
        images, labels = data
        sample_num += images.shape[0]

        pred = model(images.to(device))
        pred_classes = torch.max(pred, dim=1)[1]
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()

        loss = loss_function(pred, labels.to(device))
        accu_loss += loss

        data_loader.desc = "[valid epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step + 1),
                                                                               accu_num.item() / sample_num)

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num
def train(args,model):
    device="cuda"
    print("load dataset.........................")
    #加载数据
    dataset=MyDataSet("train/","./Data/train.csv",data_transforms["train"],"train")
    indices = list(range(len(dataset)))
    shuffle_dataset = True
    random_seed= 42
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    val=MyDataSet("test/","./Data/test.csv",data_transforms["val"],"val")
    # 验证集比例
    indices = list(range(len(val)))
    random_seed= 48
    if shuffle_dataset :
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    train_loader = DataLoader(dataset, batch_size=8, shuffle=shuffle_dataset)
    val_loader=DataLoader(val, batch_size=8, shuffle=shuffle_dataset)
    # Prepare optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=args.learning_rate,
                                momentum=0.9,
                                weight_decay=args.weight_decay)

    print("training.........................")
    #设置测试损失list,和测试acc 列表
    val_loss_list=[]
    val_acc_list=[]
    #设置训练损失list
    train_loss_list=[]
    epoch =1
#从timm库导入模型vit_base_patch16_224_in21k
    model = create_model(num_classes=3, pretrained=True).to("cuda")  
    epoch_number = 0
    for i in range(args.total_epoch):
        model.train()
        train_loss=0
        train_loss, train_acc = train_one_epoch(model=model,
                                                optimizer=optimizer,
                                                data_loader=train_loader,
                                                device=device,
                                                epoch=i)
        # validate
        val_loss, val_acc = evaluate(model=model,
                                     data_loader=val_loader,
                                     device=device,
                                     epoch=i)
        scheduler.step()
        print("train Epoch:{},loss:{},acc:{}".format(i,train_loss,train_acc))
        print("train Epoch:{},loss:{},acc:{}".format(i,val_loss,val_acc))
        # 每个epcoh保存一次模型参数
        save_model(args, model,epoch_number)
        # 每训练一个epoch,用当前训练的模型对验证集进行测试
        eval_loss, eval_acc = eval(args, model, val_loader)
        #将每一个测试集验证的结果加入列表
        """val_loss_list.append(eval_loss)
        val_acc_list.append(eval_acc)
        np.savetxt("val_loss_list(68).txt",val_loss_list)
        np.savetxt("val_acc_list(68).txt",val_acc_list)
        print("val Epoch:{},eval_loss:{},eval_acc:{}".format(epoch_number, eval_loss, eval_acc))"""
        epoch_number = epoch_number +1

def main():
    parser = argparse.ArgumentParser()
    # Required parameters
    """parser.add_argument("--dataset", choices=["cifar10", "cifar100"], default="cifar10",
                        help="Which downstream task.")"""
    parser.add_argument("--output_dir", default="./output", type=str,
                        help="The output directory where checkpoints will be written.")
    parser.add_argument("--img_size", default=224, type=int,help="Resolution size")
    parser.add_argument("--train_batch_size", default=8, type=int,
                        help="Total batch size for training.")
    parser.add_argument("--eval_batch_size", default=16, type=int,
                        help="Total batch size for eval.")
    parser.add_argument("--learning_rate", default=3e-2, type=float,
                        help="The initial learning rate for SGD.")
    parser.add_argument("--weight_decay", default=0, type=float,
                        help="Weight deay if we apply some.")
    parser.add_argument("--total_epoch", default=100, type=int,
                        help="Total number of training epochs to perform.")

    args = parser.parse_args(args=[])
    device = torch.device("cuda")
    args.device = device

    args,modle=getVisionTransformers_model(args)
    start=time.perf_counter()
    train(args,modle)
    end=time.perf_counter()
    print(end-start)

if __name__ == "__main__":
    main()




ImportError: cannot import name '_compile_and_register_class' from 'torch.jit._recursive' (/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/torch/jit/_recursive.py)

In [6]:
import timm

ImportError: cannot import name '_compile_and_register_class' from 'torch.jit._recursive' (/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/torch/jit/_recursive.py)